# 🗂️ Notebook 2: Flash Sale — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/flash-sale
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Data model

During the sale, the hot path reads/writes **Redis** only.
`stock:{item_id}` is an integer counter that starts at N.

| Key | Type | Notes |
|---|---|---|
| `stock:item-42` | int | decremented atomically |
| `user_bought:{uid}:{item}` | bool | enforce per-user cap |
| `reservation:{id}` | hash | `{user, item, qty, status=pending/paid}` |

After the sale ends, reservations are drained to the primary DB (MySQL/Postgres).

## API

```http
POST /flash/reserve      { item_id, user_id }
  → { status: "reserved", reservation_id, pay_by: 2025-... }
  → 409 if out of stock
  → 429 if rate-limited

POST /flash/pay          { reservation_id, payment_token }
```


In [ ]:
from pydantic import BaseModel
class ReserveRequest(BaseModel):
    item_id: str
    user_id: int
print(ReserveRequest(item_id="item-42", user_id=7).model_dump_json())
